# Voice Commands บน Google Colab

ฟังคำสั่งเสียงจากไมโครโฟนหรือไฟล์เสียง (hello, stop, yes, no)

## วิธีใช้

1. กดปุ่ม ▶ ที่เซลล์ด้านล่าง หรือเลือกเมนู **Runtime > Run all**
2. ถ้า Colab เตือนว่า notebook นี้ไม่ได้สร้างโดย Google ให้กด **Run anyway**
3. รอสักครู่ แอปจะแสดงใต้เซลล์นั้น

ไม่ต้องใช้ GPU เพราะโมเดลทำงานในเบราว์เซอร์ของคุณ Colab ทำหน้าที่ส่งไฟล์ของแอปเท่านั้น และไมโครโฟนที่ใช้คือไมโครโฟนของเครื่องคุณ

In [ ]:
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

from google.colab import output

REPO_URL = "https://github.com/bluebox-dev/Teachable-Machine-AnyThing.git"
REPO_DIR = Path("/content/Teachable-Machine-AnyThing")
APP_DIR = REPO_DIR / "Audio-Project" / "application"
PORT = 8002
APP_HEIGHT = "1000"  # ความสูงของแอปในหน้า notebook (พิกเซล)


def server_is_up():
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=1)
        return True
    except OSError:
        return False


# รันเซลล์นี้ซ้ำได้: ถ้ามีโปรเจกต์อยู่แล้วจะอัปเดตแทนการ clone ใหม่
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "--quiet"], check=True)
else:
    print("กำลังดาวน์โหลดโปรเจกต์จาก GitHub...")
    subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

if not server_is_up():
    subprocess.Popen(
        [sys.executable, "-m", "http.server", str(PORT), "--bind", "127.0.0.1"],
        cwd=APP_DIR,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    for _ in range(20):
        if server_is_up():
            break
        time.sleep(0.5)
    else:
        raise RuntimeError(f"เปิดเซิร์ฟเวอร์ที่พอร์ต {PORT} ไม่สำเร็จ ลองเลือก Runtime > Restart session แล้วรันอีกครั้ง")

print("แอปพร้อมใช้งานแล้ว")
output.serve_kernel_port_as_iframe(PORT, height=APP_HEIGHT)

## การใช้งานแอป

- **ไมโครโฟน:** กด **เริ่มฟัง** แล้วอนุญาตให้เบราว์เซอร์ใช้ไมโครโฟน จากนั้นลองพูดคำที่เทรนไว้
- **อัปโหลดไฟล์เสียง:** เลือกไฟล์เสียงจากเครื่องของคุณได้หลายไฟล์พร้อมกัน
- **เปลี่ยนโมเดล:** ที่การ์ด **โมเดล** วางลิงก์จาก Teachable Machine หรือเลือกไฟล์ `model.json`, `metadata.json` และ `.bin` ของคุณ

## ถ้าแอปไม่ทำงาน

| อาการ | วิธีแก้ |
| --- | --- |
| เปิดไมโครโฟนไม่ได้ | กดอนุญาตที่ไอคอนด้านซ้ายของแถบที่อยู่ในเบราว์เซอร์ แล้วรันเซลล์อีกครั้ง แนะนำให้ใช้ Google Chrome |
| แอปหายหรือขึ้นหน้าว่าง | Colab อาจตัดการเชื่อมต่อเมื่อไม่ได้ใช้งานนาน ให้รันเซลล์อีกครั้ง |
| แอปถูกตัดขอบ | เพิ่มค่า `APP_HEIGHT` ในเซลล์ แล้วรันอีกครั้ง |

ลิงก์ของแอปใช้ได้เฉพาะในหน้า notebook นี้ขณะ runtime ยังทำงานอยู่ จึงเปิดในแท็บอื่นหรือส่งให้คนอื่นไม่ได้